# 1. Executive Risk Question and Disclaimer

How much could the portfolio lose over the next valid valuation interval, what drives the loss, and does the model's out-of-sample exception behaviour look acceptable?

> Educational model — not approved for regulatory capital or live trading limits. The bundled snapshot is **SYNTHETIC DATA — NOT FOR RESUME RESULTS**.

# 2. Environment Setup
The notebook installs and calls the same tested package used by the CLI. In a clean Colab runtime, store a public clone URL in the `MARKET_RISK_REPO_URL` Colab Secret.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

candidates = [Path.cwd(), Path.cwd() / 'market-risk-engine', Path('/content/market-risk-engine')]
ROOT = next((path for path in candidates if (path / 'pyproject.toml').exists()), None)
if ROOT is None:
    try:
        from google.colab import userdata
        repo_url = userdata.get('MARKET_RISK_REPO_URL')
    except Exception:
        repo_url = None
    repo_url = repo_url or 'https://github.com/sheeeeshba/market-risk-engine.git'
    if not repo_url:
        raise RuntimeError('Repository URL is unavailable. Set MARKET_RISK_REPO_URL to a public fork and rerun.')
    ROOT = Path('/content/market-risk-engine')
    subprocess.run(['git', 'clone', repo_url, str(ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
print(f'Project root: {ROOT}')

# 3. Configuration
Change mode through the `MARKET_RISK_DATA_MODE` environment variable. Live mode also requires the `FRED_API_KEY` Secret/environment variable.

In [ ]:
DATA_MODE = os.environ.get('MARKET_RISK_DATA_MODE', 'snapshot')
CONFIG_PATH = ROOT / 'config/model_config.yaml'
print({'data_mode': DATA_MODE, 'config': str(CONFIG_PATH)})

# 4. Position Validation
The default catalog allocation is resolved with automatic residual cash; funded weights sum to 100% and the FX overlay is validated separately.

In [ ]:
from market_risk.portfolio import initial_position_snapshot
from market_risk.portfolio_builder import load_default_portfolio
from market_risk.validation import validate_portfolio_config
catalog, library, allocation, portfolio_config = load_default_portfolio(ROOT)
print({'catalog_instruments': len(catalog.instruments), 'preset': allocation.preset_key})
print(validate_portfolio_config(portfolio_config))
initial_position_snapshot(portfolio_config)

# 5. Market Data Retrieval or Snapshot Loading
The deep pipeline interface performs data loading, accounting, risk, backtesting, stresses, figures, reporting, and evidence recording.

In [ ]:
from market_risk.engine import run_pipeline
artifacts = run_pipeline(CONFIG_PATH, DATA_MODE)
print(json.dumps(artifacts, indent=2))

# 6. Data Quality and Calendar Alignment
Market-price/FX levels are never filled. Treasury rate fills are limited and flagged in live mode; snapshot provenance and factor hash are shown below.

In [ ]:
metadata_path = ROOT / 'data/snapshots/synthetic_demo_metadata.json'
if metadata_path.exists(): print(json.dumps(json.loads(metadata_path.read_text()), indent=2))
live_quality = ROOT / 'data/processed/latest_live_data_quality.csv'
if DATA_MODE == 'live' and live_quality.exists():
    import pandas as pd
    display(pd.read_csv(live_quality).tail())

# 7. Risk-Factor Shocks
Returns are decimal simple returns; Treasury changes are absolute decimal yield changes.

In [ ]:
import pandas as pd
factor_file = ROOT / ('data/processed/latest_live_factors.csv' if DATA_MODE == 'live' else 'data/snapshots/synthetic_demo_factors.csv')
factors = pd.read_csv(factor_file, index_col='portfolio_date', parse_dates=True)
display(factors.tail())
display(factors.describe().T[['mean', 'std', 'min', 'max']])

# 8. Portfolio and Hypothetical P&L
The table reconciles daily position P&L to portfolio P&L and funded market value to NAV.

In [ ]:
daily = pd.read_csv(ROOT / 'outputs/tables/daily_portfolio_pnl.csv', index_col=0, parse_dates=True)
display(daily.tail())
assert (daily.position_pnl_sum - daily.portfolio_pnl).abs().max() < 0.01

# 9. Current Historical VaR/ES
Historical ES uses exact fractional tail mass and equal boundary-tie weights.

In [ ]:
risk = pd.read_csv(ROOT / 'outputs/tables/current_risk.csv')
display(risk[risk.model == 'Historical'])

# 10. Current Parametric VaR/ES
The primary model uses zero mean and sample covariance.

In [ ]:
display(risk[risk.model == 'Parametric Normal'])

# 11. Current Monte Carlo VaR/ES
The model uses deterministic zero-mean Normal factor draws and full duration-convexity revaluation.

In [ ]:
display(risk[risk.model == 'Monte Carlo Normal'])
display(pd.read_csv(ROOT / 'outputs/tables/monte_carlo_convergence.csv'))

# 12. Rolling Forecasts
Each forecast ends at `t` and is compared with `t_next`; future timestamps hard-fail validation.

In [ ]:
forecasts = pd.read_csv(ROOT / 'outputs/tables/rolling_forecasts.csv', parse_dates=['forecast_date','realized_date','expected_next_date','max_input_date'])
display(forecasts.head(9))

# 13. Backtesting
Non-rejection is not proof of correctness; read low-power warnings.

In [ ]:
display(pd.read_csv(ROOT / 'outputs/tables/backtesting_scorecard.csv'))

# 14. Risk Contributions
Component VaR and Historical ES contributions are additive; negative values are retained.

In [ ]:
display(pd.read_csv(ROOT / 'outputs/tables/risk_contributions.csv'))

# 15. Hypothetical Stress Tests
Six scenarios preserve gains as negative losses and reconcile to positions.

In [ ]:
display(pd.read_csv(ROOT / 'outputs/tables/stress_summary.csv'))

# 16. Historical Crisis Replays
The bundled replay archive is artificial and tests engineering only; it must be replaced before historical claims.

In [ ]:
display(pd.read_csv(ROOT / 'outputs/tables/crisis_replay_summary.csv'))

# 17. Volatility and Correlation Stress
Distribution stress changes risk estimates but has zero immediate deterministic P&L.

In [ ]:
display(pd.read_csv(ROOT / 'outputs/tables/volatility_correlation_stress.csv'))

# 18. Automated Risk Report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path(artifacts['report']).read_text()))
print('Report:', artifacts['report'])
print('Manifest:', artifacts['manifest'])

# 19. Conclusions, Limitations, and Next Steps
The complete engineering path is reproducible offline. The next evidence gate is a permitted real-data snapshot, followed by a fresh Colab run from a published repository. Until then, do not use the synthetic metrics, crisis values, screenshot, or resume draft as historical evidence.